# A3.3.1: SQL data language types

*IB Computer Science 2027, topic A3 Databases, A3.3 Database programming.* Core for SL and HL. Command term: **outline**. Builds on A3.2.4.

*Building a shelf is not the same job as putting books on it. SQL keeps those two jobs in two separate language types, and knowing which is which is the whole of this statement.*

**How to use this notebook**
- Click a code cell and press **Shift+Enter** to run it. Run the cells **in order, from the top**.
- A cell that starts with `%%sql` holds plain SQL. A cell with `<<` after `%%sql` saves its result so the check cell can look at it. A cell that creates or changes things shows an empty box. That is normal.
- A **TODO** cell is yours. Each starts with a placeholder line `SELECT 'write your SQL here' AS todo;`. Delete that line when you have written your own SQL.
- The **check** cells print ✓ when you are right and ✗ with the difference when you are not.
- The database lives in memory. If you restart the kernel, run the cells again from the top.

Notes adapted from the ExamEdge page https://examedge.app/learn/ib/a3-3-1

In [1]:
# Helper functions for the check cells. Run this once. You do not need to read it.
def _rows(var):
    if var not in globals():
        return None
    return [tuple(r) for r in globals()[var]]

def _norm(rows, places=2):
    return [tuple(round(v, places) if isinstance(v, float) else v for v in r) for r in rows]

def _compare(name, got, want, ordered):
    g, w = _norm(got), _norm(want)
    ok = (g == w) if ordered else (sorted(g, key=repr) == sorted(w, key=repr))
    if ok:
        print("✓", name, "is correct.")
    else:
        print("✗", name, "is not right yet.")
        print("   You returned", len(g), "row(s):", g[:6])
        print("   Expected    ", len(w), "row(s):", w[:6])

def check_rows(name, var, want, ordered=False):
    # Compares the rows saved by a "%%sql var <<" cell with the expected rows.
    got = _rows(var)
    if got is None:
        print("✗", name, ": there is no result yet. Run your SQL cell first.")
        return
    _compare(name, got, want, ordered)

def check_db(name, sql, want, ordered=False):
    # Runs a question against the practice database and compares the rows.
    try:
        got = [tuple(r) for r in db.execute(sql).fetchall()]
    except Exception as e:
        print("✗", name, ": my check could not run:", e)
        return
    _compare(name, got, want, ordered)

def check_choice(choice, correct, why_right, why_wrong):
    c = str(choice).strip().upper()
    if c not in "ABCD" or c == "":
        print("Set choice to A, B, C or D, then run this cell again.")
    elif c == correct:
        print("✓ Correct.", why_right)
    else:
        print("✗ Not quite.", why_wrong)

def check_dict(name, got, want):
    bad = [k for k in want if str(got.get(k, "")).strip().upper() != want[k]]
    if not bad:
        print("✓", name, "all correct.")
    else:
        print("✗", name, ":", len(bad), "of", len(want), "not right yet:")
        for k in bad:
            print("    ", k)

In [2]:
# Install the SQL add-on into THIS kernel if it is missing (one time only). (Given.)
import importlib.util, subprocess, sys
if importlib.util.find_spec("sql") is None:
    print("Installing the SQL add-on (one time, about a minute)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"])
    importlib.invalidate_caches()

import sqlite3
db = sqlite3.connect(":memory:")

%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = False
%sql db

## 1. Two jobs, one language

SQL is really two different jobs written in the same syntax.

- **Data definition language (DDL)** never touches a single row of data. It brings a container (a table, view, index or constraint) into existence, reshapes it, or removes it.
- **Data manipulation language (DML)** never redraws the container. It works with what is *inside* it: reading rows, adding rows, changing rows or removing rows. The table must already exist, built by DDL first.

| DDL | What it does | DML | What it does |
|---|---|---|---|
| `CREATE` | makes a new table, view or index | `SELECT` | reads rows back out |
| `PRIMARY KEY` | marks a table's primary key | `INSERT` | adds a new row |
| `FOREIGN KEY ... REFERENCES ...` | links to another table | `UPDATE` | changes values in an existing row |
| `ALTER` | reshapes a table that already has data | `DELETE` | removes a row |
| `DROP` | deletes a table, index or view entirely | | |

## 2. Worked example: an animal shelter

We shape the table first (DDL) and then fill it (DML). Run each cell in order and watch what it does.

**Step 1 (DDL).** Create the `Shelter` table, then the `Pet` table with a primary key and a foreign key.

In [3]:
%%sql
CREATE TABLE Shelter (
    ShelterID   INTEGER PRIMARY KEY,
    ShelterName TEXT NOT NULL
);

++
||
++
++

In [4]:
%%sql
CREATE TABLE Pet (
    PetID     INTEGER PRIMARY KEY,
    PetName   TEXT NOT NULL,
    Species   TEXT NOT NULL,
    Age       INTEGER,
    ShelterID INTEGER,
    FOREIGN KEY (ShelterID) REFERENCES Shelter (ShelterID)
);

++
||
++
++

Both tables exist but are empty. Nothing in them has been touched, because DDL only deals with structure.

**Step 2 (DML).** Now add rows with `INSERT`.

In [5]:
%%sql
INSERT INTO Shelter (ShelterID, ShelterName) VALUES (1, 'Harbour Rescue');
INSERT INTO Shelter (ShelterID, ShelterName) VALUES (2, 'Hillside Haven');
INSERT INTO Pet (PetID, PetName, Species, Age, ShelterID) VALUES (1, 'Biscuit', 'Dog', 4, 1);
INSERT INTO Pet (PetID, PetName, Species, Age, ShelterID) VALUES (2, 'Pepper', 'Cat', 3, 1);
INSERT INTO Pet (PetID, PetName, Species, Age, ShelterID) VALUES (3, 'Juno', 'Rabbit', 1, 2);
INSERT INTO Pet (PetID, PetName, Species, Age, ShelterID) VALUES (4, 'Rocky', 'Dog', 7, 2);

++
||
++
++

In [6]:
%%sql
SELECT * FROM Pet;

PetID,PetName,Species,Age,ShelterID
1,Biscuit,Dog,4,1
2,Pepper,Cat,3,1
3,Juno,Rabbit,1,2
4,Rocky,Dog,7,2


**Step 3 (DML).** `UPDATE` changes a value that is already in a row. The table's columns do not change at all.

In [7]:
%%sql
UPDATE Pet SET Age = 5 WHERE PetID = 1;
SELECT * FROM Pet;

PetID,PetName,Species,Age,ShelterID
1,Biscuit,Dog,5,1
2,Pepper,Cat,3,1
3,Juno,Rabbit,1,2
4,Rocky,Dog,7,2


**Step 4 (DDL).** `ALTER TABLE` changes the table's **structure**. Look at the new column `Adopted`. Every existing row gets `NULL` there, because ALTER does not touch the rows.

In [8]:
%%sql
ALTER TABLE Pet ADD COLUMN Adopted BOOLEAN;
SELECT * FROM Pet;

PetID,PetName,Species,Age,ShelterID,Adopted
1,Biscuit,Dog,5,1,None
2,Pepper,Cat,3,1,None
3,Juno,Rabbit,1,2,None
4,Rocky,Dog,7,2,None


**Step 5 (DML).** `DELETE` removes a row, again without changing the structure.

In [9]:
%%sql
DELETE FROM Pet WHERE PetID = 4;
SELECT * FROM Pet;

PetID,PetName,Species,Age,ShelterID,Adopted
1,Biscuit,Dog,5,1,None
2,Pepper,Cat,3,1,None
3,Juno,Rabbit,1,2,None


**Step 6 (DDL).** `DROP` removes the container itself. We will do this on a spare table so we keep `Pet`.

In [10]:
%%sql
CREATE TABLE Spare (x INTEGER);
DROP TABLE Spare;
SELECT name FROM sqlite_master WHERE type = 'table';

name
Shelter
Pet


`Spare` is gone, but `Shelter` and `Pet` are still there.

> **Why not classify by the word "change"?** `ALTER` changes a table and `UPDATE` changes a record, so both sound like "changing the database". The real test is **what** is changing: the structure itself (DDL) or a value inside an unchanged structure (DML).

## 3. Your turn

**TODO 1 (DDL).** We will rebuild the shelter in your own tables. First reset the database so you start clean, then write `CREATE TABLE` for a table called `Dog` with the columns `DogID` (the primary key), `DogName` (text) and `Age` (whole number).

In [ ]:
db.executescript("DROP TABLE IF EXISTS Pet; DROP TABLE IF EXISTS Shelter; DROP TABLE IF EXISTS Dog;")
print("Database cleared.")

In [ ]:
%%sql
-- TODO 1: CREATE TABLE Dog (DogID ..., DogName ..., Age ...) with DogID as the primary key.
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_db("Dog columns", "SELECT name FROM pragma_table_info('Dog') ORDER BY cid", [("DogID",), ("DogName",), ("Age",)], ordered=True)
check_db("Dog primary key", "SELECT name FROM pragma_table_info('Dog') WHERE pk = 1", [("DogID",)])

**TODO 2 (DML).** Add three dogs: Biscuit (age 4), Rocky (age 7) and Juno (age 1). Use `INSERT INTO Dog (DogID, DogName, Age) VALUES (...)`.

In [ ]:
%%sql
-- TODO 2: three INSERT statements, one per dog.
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_db("Three dogs", "SELECT DogName, Age FROM Dog", [("Biscuit", 4), ("Rocky", 7), ("Juno", 1)])

**TODO 3 (DDL).** Add a new column `Adopted` (type `BOOLEAN`) to the `Dog` table using `ALTER TABLE ... ADD COLUMN ...`.

**TODO 4 (DML).** Then set `Adopted` to `1` for Rocky only, using `UPDATE ... SET ... WHERE ...`.

In [ ]:
%%sql
-- TODO 3 and 4: one ALTER TABLE statement, then one UPDATE statement.
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_db("Adopted column added", "SELECT name FROM pragma_table_info('Dog') WHERE name = 'Adopted'", [("Adopted",)])
check_db("Only Rocky adopted", "SELECT DogName FROM Dog WHERE Adopted = 1", [("Rocky",)])

**TODO 5: sort the statements.** For each statement write `"DDL"` or `"DML"` in the dictionary, then run the check.

In [ ]:
answers = {
    "DROP TABLE Pet;": "?",
    "INSERT INTO Pet VALUES (5, 'Milo', 'Cat', 2);": "?",
    "CREATE INDEX idx_species ON Pet (Species);": "?",
    "SELECT * FROM Pet;": "?",
    "ALTER TABLE Pet ADD COLUMN Adopted BOOLEAN;": "?",
    "UPDATE Pet SET Age = 3 WHERE PetID = 5;": "?",
    "DELETE FROM Pet WHERE PetID = 5;": "?",
}
check_dict("TODO 5", answers, {k: v for k, v in zip(answers, ["DDL", "DML", "DDL", "DML", "DDL", "DML", "DML"])})

## 4. Check your understanding

Which statement is DDL: **A** `ALTER TABLE PET ADD COLUMN Adopted BOOLEAN;`, **B** `UPDATE PET SET Adopted = TRUE WHERE PetID = 4;`, **C** both are DDL, or **D** neither, both are DML?

In [ ]:
choice = "?"   # type A, B, C or D
check_choice(choice, "A", "ALTER changes the table's structure (a column that did not exist before), so it is DDL. UPDATE changes a value already stored, so it is DML.",
             "Think about what changes. UPDATE changes a value inside an existing row, the structure stays the same, so that is DML.")

## 5. Examiner note: a common mistake

Weak answer: *"UPDATE is DDL because it changes the database."* It does change something, but it changes a stored **value**, not the table's structure, so it is DML.

Better answer: *"UPDATE is DML: it modifies a value already stored in an existing row. The table's columns and structure stay exactly as they were before and after."* It names the type **and** explains why by pointing at what actually changed.

## 6. Revision questions

**Q1. Outline the difference between DDL and DML.** [2 marks]

Double-click this box and type your answer here, then press Shift+Enter.

> (your answer)

**Q2. State whether each of the following is DDL or DML, with a reason: `DROP TABLE Pet;`, `INSERT INTO Pet VALUES (5, 'Milo', 'Cat', 2);`, `CREATE INDEX idx_species ON Pet (Species);`** [3 marks]

Double-click this box and type your answer here, then press Shift+Enter.

> (your answer)

**Q3. A developer writes `ALTER TABLE Pet ADD FOREIGN KEY (ShelterID) REFERENCES Shelter (ShelterID);`. Explain why this is DDL rather than DML, and outline what would happen if a matching ShelterID did not exist.** [4 marks]

Double-click this box and type your answer here, then press Shift+Enter.

> (your answer)

## 7. Linking question (connects to B2)

*How does database programming in SQL differ from programming in a high-level language?* SQL statements like `CREATE`, `SELECT` or `UPDATE` describe **what** result is wanted and the DBMS works out how. A language like Python describes **how** to get there, step by step. DDL and DML are this declarative style in action: no loop or condition is needed to add a record or change a table.